# Lecture 5 self-check: Python packaging and continuous integration: answers

The answers to the Lecture 5 self-check, with a short explanation for each, including why the tempting wrong options are wrong. You can check the packaging ones by running the code yourself.

### Question 1

`add.py` contains

```python
import sys

print(sys.argv[1] + sys.argv[2])
```

What does `python add.py 3 4` print?

- **(A)** Nothing: it stops with `IndexError: list index out of range`
- **(B)** `7`
- **(C)** `34`
- **(D)** `add.py3`

**Answer: (C)**

`sys.argv` is the list of the command-line arguments, as strings: `sys.argv[0]` is the script's name, `'add.py'`, and the arguments start at `sys.argv[1]`. Here `sys.argv` is `['add.py', '3', '4']`, so `sys.argv[1] + sys.argv[2]` joins two strings: `'3' + '4'` is `'34'`. To add them as numbers, convert them first: `int(sys.argv[1]) + int(sys.argv[2])` is `7` (or let `argparse` do it, with `type=int`). `python` itself is not in the list, so `sys.argv[1]` is not `'add.py'`; and because the arguments start at index 1, `sys.argv[2]` exists.


### Question 2

A folder holds two files. `shout.py`:

```python
print("A:", __name__)


def shout(s):
    return s.upper()


if __name__ == "__main__":
    print("B:", shout("hi"))
```

`main.py`:

```python
import shout

print("C:", shout.shout("ok"))
```

What does `python main.py` print?

- **(A)** `A: shout`, then `C: OK`
- **(B)** Only `C: OK`
- **(C)** `A: shout.py`, then `C: OK`
- **(D)** `A: __main__`, then `B: HI`, then `C: OK`

**Answer: (A)**

Importing a module runs its code, from the top, the first time it is imported: so `import shout` runs the `print` at the top of `shout.py`, and defines `shout`. In a module that is imported, `__name__` is the module's name, `'shout'` (not the file's name), so that line prints `A: shout`, and the test `__name__ == "__main__"` is false: the `B:` line doesn't run. Only the file you run, here `main.py`, has `__name__` equal to `"__main__"`. Then `main.py` prints `C: OK`. Running `python shout.py` would print `A: __main__` and `B: HI`: an `if __name__ == "__main__":` block is for code that should run when the file is run as a script, but not when it is imported.


### Question 3

A project folder looks like this:

```text
project/
├── helpers.py
├── report.py
└── scripts/
    └── run.py
```

`helpers.py` defines a function `greet()` that returns `"hello"`, and `report.py` and `scripts/run.py` both contain

```python
import helpers

print(helpers.greet())
```

`PYTHONPATH` is not set, and nothing in `project/` has been installed with `pip`. In a terminal in `project/`, you run `python report.py`, then `python scripts/run.py`. What happens?

- **(A)** `report.py` and `scripts/run.py` both raise `ModuleNotFoundError`
- **(B)** `report.py` prints `hello`; `scripts/run.py` raises `ModuleNotFoundError`
- **(C)** `report.py` and `scripts/run.py` both print `hello`

**Answer: (B)**

When you run a script, Python puts the folder that contains *the script* at the front of its search path (`sys.path[0]`), not the folder you are in. `report.py` is in `project/`, next to `helpers.py`, so its import works. `scripts/run.py` is in `scripts/`, which has no `helpers.py`, and the standard library and `site-packages` don't have it either, so its import fails, even though you ran it from `project/`, where `helpers.py` is. (In an interactive session, such as a notebook, it is the current folder that is searched.) Your own modules don't need installing to be imported, only to be on the search path. The usual way to share code between scripts in different folders is to make it a package and install it with `pip install -e .`, so that it can be imported from anywhere.


### Question 4

You release your package under the GPL (GNU General Public License). Someone changes your code and distributes their changed version. What does the licence require of them?

- **(A)** They must release their version as open source, under the same licence
- **(B)** They need only keep your copyright notice, and may keep their changes closed source
- **(C)** They must send their changes back to you, the original author
- **(D)** They may not distribute a changed version at all: only you may

**Answer: (A)**

The GPL is a *copyleft* licence: anyone may use, change and share the code, but a changed version that they distribute must also be open source, under the GPL (the notes: it "requires derivatives to also be open-source under the same license"). The permissive licences, MIT and BSD, are the ones that allow almost unrestricted use, including in closed-source software. An open-source licence exists so that people *can* change and share your code without asking you each time: the licence is the permission, and its conditions say what they must do in return. The GPL's condition is about the people who receive the changed version, who must get its source under the GPL; it doesn't require anyone to send their changes back to you.


### Question 5

A package's `pyproject.toml` has

```toml
dependencies = ["numpy~=1.21"]
```

The NumPy versions available are `1.21.0`, `1.21.6`, `1.26.4` and `2.5.3`. Which of these is the newest version that this dependency allows?

- **(A)** `2.5.3`
- **(B)** `1.21.0`
- **(C)** `1.21.6`
- **(D)** `1.26.4`

**Answer: (D)**

`~=` is the *compatible release* specifier: `~=1.21` means `>=1.21` and `<2.0` (the last number given may go up; the ones before it stay the same). So `1.21.0`, `1.21.6` and `1.26.4` are allowed, and the newest is `1.26.4`; `2.5.3` is a new major version, and is not allowed. With one more number, `~=1.21.0` would allow only the 1.21 releases, up to `1.21.6`; `>=1.21` alone would allow `2.5.3`; and `==1.21.0` only `1.21.0`.


### Question 6

You install your package `mycoolpackage` with `pip install -e .`, from its folder. In a Python session started in another folder, you run `import mycoolpackage.cool_module` and call `mycoolpackage.cool_module.greet()`. Then you edit `mycoolpackage/cool_module.py` so that `greet()` returns a different message. What does `mycoolpackage.cool_module.greet()` return (i) when you just call it again in that session, without reloading anything, and (ii) in a new session?

- **(A)** (i) the new message; (ii) the new message
- **(B)** (i) the old message; (ii) the new message
- **(C)** (i) the old message; (ii) the old message

**Answer: (B)**

`pip install -e .` installs the package in *editable* mode: Python imports it from your folder, so a new session runs your edited code without reinstalling, and (ii) returns the new message. But a session keeps a module it has already imported: a module's code runs only on its first import, so (i), in the session that imported it before the edit, still returns the old message. To see the edit there, run `importlib.reload(mycoolpackage.cool_module)` or restart. A plain `pip install .` would behave differently: it copies the package into the environment's `site-packages`, and Python imports that copy, so even a new session would return the old message until you install it again.


### Question 7

The package `stats_tools` lists NumPy as its dependency in `pyproject.toml`, and `pytest` and `flake8` as `dev` extras. Its tests pass on your machine. GitHub's runners don't come with `pytest` or NumPy. In the two workflows below, every job runs `actions/checkout` and `actions/setup-python` first, then the steps listed, in order.

- (i) Job `setup`: `pip install -e ".[dev]"`. Job `test` (`needs: setup`): `pytest tests/`.
- (ii) One job, `test`: `pip install -e .`, then `pytest tests/`.

Does the `pytest tests/` step pass on GitHub in each?

- **(A)** (i) passes; (ii) passes
- **(B)** (i) passes; (ii) fails
- **(C)** (i) fails; (ii) passes
- **(D)** (i) fails; (ii) fails

**Answer: (D)**

Both fail, with `pytest: command not found`. Each job runs on a fresh runner of its own, with only what that job's own steps install. In (i), `needs: setup` only makes `test` wait until `setup` has succeeded: nothing that `setup` installed reaches `test`'s runner, so `pytest` was never installed there. To fix it, install in the job that runs the tests. In (ii), `pip install -e .` installs the package and NumPy, but not the `dev` extras, which come only with `pip install -e ".[dev]"`.


### Question 8

Your repository's default branch is `main`, and its workflow `.github/workflows/flake8.yml` starts

```yaml
name: flake8

on:
  push:
    branches:
      - main
  pull_request:
    branches:
      - main
```

In which of these situations does the workflow run?

- **(A)** You push a commit to a new branch, `fix-sum`, with no pull request open
- **(B)** You push another commit to branch `fix-sum`, whose pull request into `main` is already open
- **(C)** You open a pull request from `main` into another branch, `dev`
- **(D)** You open a pull request from `fix-sum` into another branch, `dev`

**Answer: (B)**

The `on:` block says when the workflow runs: on a push *to `main`*, and on a pull request *into `main`*, both when it is opened and each time it is updated by a push to its branch. So pushing to `fix-sum` while its pull request into `main` is open runs the workflow (for the pull request). A push to `fix-sum` with no pull request open doesn't: the `branches:` under `push` limits it to pushes to `main` (a bare `on: push` would run for a push to any branch). A pull request into `dev` doesn't match the `branches:` under `pull_request`, which is about the branch the pull request goes *into*, not the branch it comes from: so a pull request from `main` into `dev` doesn't run it either.


### Question 9

A workflow has two jobs:

```yaml
name: checks

on:
  push:
    branches:
      - main

jobs:
  lint:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v7

    - name: Set up Python 3.14
      uses: actions/setup-python@v7
      with:
        python-version: "3.14"

    - name: Lint with flake8
      run: |
        pip install flake8
        flake8 .

  test:
    needs: lint
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v7

    - name: Set up Python 3.14
      uses: actions/setup-python@v7
      with:
        python-version: "3.14"

    - name: Install and test
      run: |
        pip install -e ".[dev]"
        pytest tests/
```

The package's tests all pass, but `flake8` finds a style problem, so the `lint` job fails. What happens to the `test` job?

- **(A)** It doesn't run: GitHub shows it as skipped
- **(B)** It runs once `lint` has finished, and passes
- **(C)** It starts at the same time as `lint`, and is cancelled when `lint` fails
- **(D)** It runs once `lint` has finished, and fails, because `lint` failed

**Answer: (A)**

Jobs run in parallel unless `needs:` says otherwise. `needs: lint` makes `test` wait for `lint`, and run only if `lint` *succeeds*. Here `lint` failed, so `test` never starts, and GitHub shows it as skipped (a job-level `if: ${{ always() }}` would make it run anyway). It doesn't run and then fail, and it wasn't running alongside `lint` to be cancelled: cancelling the rest when one job fails is what `fail-fast` does in a matrix. Without the `needs:` line, `test` would have run at the same time as `lint`, and passed.


### Question 10

The package `stats_tools` has this `pyproject.toml`:

```toml
[build-system]
requires = ["setuptools>=77.0.3"]
build-backend = "setuptools.build_meta"

[project]
name = "stats_tools"
version = "0.1.0"
dependencies = ["numpy"]

[project.optional-dependencies]
dev = ["pytest", "flake8"]
```

Its module `stats_tools/summary.py` starts

```python
import numpy as np
from scipy.ndimage import uniform_filter1d
```

and the last two steps of its `pytest-unit-tests.yml` workflow (after `actions/checkout` and `actions/setup-python`) are

```yaml
    - name: Install the package and its dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -e ".[dev]"

    - name: Test with pytest
      run: |
        pytest tests/
```

The tests pass on your machine, in `mpm2026`, but on GitHub the job fails with `ModuleNotFoundError: No module named 'scipy'`. Which change makes the job pass?

- **(A)** Add an `environment.yml` that lists `numpy` and `scipy`
- **(B)** Add a `requirements.txt` that lists `numpy` and `scipy`
- **(C)** Add `"scipy"` to `requires` in the `[build-system]` table of `pyproject.toml`
- **(D)** Change `dependencies` in `pyproject.toml` to `["numpy", "scipy"]`

**Answer: (D)**

The runner is a fresh machine: it has only what the workflow installs, and `pip install -e ".[dev]"` installs the package with the dependencies its `pyproject.toml` declares (`numpy`) and its `dev` extras. SciPy is in `mpm2026`, which is why the tests pass on your machine, but it was never declared. Adding it to `dependencies` fixes the job, and means that anyone who installs the package with `pip` gets SciPy too. Nothing in the workflow reads an `environment.yml` (that is for `conda env create`) or a `requirements.txt` (that needs `pip install -r requirements.txt`). And `requires` in `[build-system]` lists what is needed to *build* the package: `pip` installs it in a temporary build environment, not alongside the package.


### Question 11

How many jobs does a push run, with this workflow?

```yaml
name: pytest-matrix

on: push

jobs:
  pytest:
    name: ${{ matrix.os }}, Python ${{ matrix.python-version }}
    runs-on: ${{ matrix.os }}

    strategy:
      matrix:
        os: [ubuntu-latest, windows-latest, macos-latest]
        python-version: ["3.12", "3.14"]

    steps:
    - uses: actions/checkout@v7

    - name: Set up Python ${{ matrix.python-version }}
      uses: actions/setup-python@v7
      with:
        python-version: ${{ matrix.python-version }}

    # ... then install and test, as in pytest-unit-tests.yml
```

- **(A)** 5
- **(B)** 1
- **(C)** 6
- **(D)** 3

**Answer: (C)**

A matrix runs the same job once for each *combination* of its values, each on its own runner, in parallel: 3 operating systems × 2 Python versions makes 6 jobs, each named by the job's `name:` line (e.g. `windows-latest, Python 3.12`). That isn't one job per value listed (5), one job per operating system that tries each Python version in turn (3), or one job that works through the lists (1). `exclude` (and `include`) can remove (or add) particular combinations.


### Question 12

A step in your workflow uploads its results with an API token, which it reads from the environment variable `API_TOKEN`. Your repository is public, and the workflow runs on pushes to `main`. In which of these set-ups does the step get the token, without anyone who can see your repository being able to read it?

- **(A)** The token is in the workflow file itself; the step's `env:` has `API_TOKEN: <the token>`
- **(B)** The token is a repository secret named `API_TOKEN`, like the variable; nothing in the workflow refers to it
- **(C)** The token is a repository secret, `MY_API_TOKEN`; the step's `env:` has `API_TOKEN: ${{ secrets.MY_API_TOKEN }}`
- **(D)** The token is in `token.txt`, which `.gitignore` keeps out of git; the step reads that file

**Answer: (C)**

A repository secret (added under `Settings` → `Secrets and variables` → `Actions`) is stored by GitHub, not in the repository, and GitHub shows it as `***` if it appears in a log. A step gets it only through the expression `${{ secrets.MY_API_TOKEN }}`, here put in the step's environment as `API_TOKEN`; a secret that the workflow never mentions isn't an environment variable of the steps, even with the same name. Writing the token in the workflow does give it to the step, but the workflow is a file in your public repository, which anyone can read. And a file that `.gitignore` keeps out of the repository never reaches the runner: it is a fresh machine with only the committed files (from `actions/checkout`).
